## Summary

This notebook provides comprehensive SHAP explanations for both stages of the FMD early warning system:

### Stage 1 (Outbreak Prediction)
- **Model:** Logistic Regression with class weighting
- **SHAP Explainer:** LinearExplainer
- **Visualizations:** Summary plot (beeswarm), Feature importance bar chart, Single prediction waterfall
- **Insight:** Shows which environmental and livestock features contribute most to outbreak risk

### Stage 2 (Severity Classification)
- **Model:** Random Forest (3-class: LOW, MEDIUM, HIGH)
- **SHAP Explainer:** TreeExplainer
- **Visualizations:** Summary plot (beeswarm), Feature importance bar chart
- **Insight:** Reveals which features distinguish severity levels in confirmed outbreaks

### Key Outputs
1. **4 SHAP visualization plots** saved to `plots/08_shap/`
2. **2 CSV files** with mean absolute SHAP values per feature
3. **Early warning report** for Anuradhapura January 2024 with actionable recommendations

### Recommendations
- Use SHAP plots for stakeholder communication about model interpretability
- Reference early warning report for district-level decision making
- Monitor top features (livestock density, geographic location, weather) as key indicators

In [50]:
# CELL 10: Save SHAP Values as CSV

print('\n' + '='*60)
print('CELL 10: Exporting SHAP Values to CSV')
print('='*60)

# Stage 1 SHAP values (CLIMATE FEATURES ONLY - excluding district_enc)
shap_df_stage1 = pd.DataFrame({
    'feature': climate_feature_names,
    'mean_abs_shap': mean_abs_shap_stage1
}).sort_values('mean_abs_shap', ascending=False)

stage1_csv_path = PROC_DIR / 'stage1_shap_values.csv'
shap_df_stage1.to_csv(stage1_csv_path, index=False)
print(f'✅ Saved Stage 1 SHAP values: {stage1_csv_path}')
print(f'   {len(shap_df_stage1)} climate/environmental features (district_enc EXCLUDED)')
print(f'   Top 5:')
for idx, row in shap_df_stage1.head(5).iterrows():
    print(f'      {row["feature"]:30s}: {row["mean_abs_shap"]:.6f}')

# Stage 2 SHAP values - handle multiclass case
if isinstance(mean_abs_shap_stage2, np.ndarray) and mean_abs_shap_stage2.ndim == 1:
    mean_shap_flat = mean_abs_shap_stage2
elif isinstance(mean_abs_shap_stage2, np.ndarray):
    mean_shap_flat = np.mean(np.abs(mean_abs_shap_stage2), axis=0) if mean_abs_shap_stage2.ndim > 1 else mean_abs_shap_stage2.flatten()
else:
    mean_shap_flat = mean_abs_shap_stage2

shap_df_stage2 = pd.DataFrame({
    'feature': stage2_feature_cols,
    'mean_abs_shap': mean_shap_flat
}).sort_values('mean_abs_shap', ascending=False)

stage2_csv_path = PROC_DIR / 'stage2_shap_values.csv'
shap_df_stage2.to_csv(stage2_csv_path, index=False)
print(f'\n✅ Saved Stage 2 SHAP values: {stage2_csv_path}')
print(f'   {len(shap_df_stage2)} features')
print(f'   Top 5:')
for idx, row in shap_df_stage2.head(5).iterrows():
    print(f'      {row["feature"]:30s}: {row["mean_abs_shap"]:.6f}')

print('\n' + '='*60)
print('✅ SHAP EXPLAINABILITY ANALYSIS COMPLETE')
print('='*60)
print(f'\n⚠️  IMPORTANT NOTES:')
print(f'   • Stage 1 SHAP excludes district_enc (categorical identifier)')
print(f'   • Stage 1 model should be retrained WITHOUT district_enc')
print(f'   • All visualizations show CLIMATE/ENVIRONMENTAL features only')
print(f'\nGenerated Artifacts:')
png_count = len(list(PLOT_DIR.glob('*.png')))
print(f'  Plots: {png_count} PNG files')
for png in sorted(PLOT_DIR.glob('*.png')):
    print(f'    - {png.name}')
print(f'  CSVs: stage1_shap_values.csv (climate only), stage2_shap_values.csv')
print(f'  Report: early_warning_report_anuradhapura_jan2024.txt')


CELL 10: Exporting SHAP Values to CSV
✅ Saved Stage 1 SHAP values: ..\data\processed\stage1_shap_values.csv
   21 climate/environmental features (district_enc EXCLUDED)
   Top 5:
      cos_month                     : 0.596302
      r3h                           : 0.556246
      lat                           : 0.316275
      humidity                      : 0.286263
      sin_month                     : 0.282171

✅ Saved Stage 2 SHAP values: ..\data\processed\stage2_shap_values.csv
   21 features
   Top 5:
      buffalo_density               : 0.053643
      lat                           : 0.048202
      lon                           : 0.034327
      livestock_density             : 0.031607
      wind_speed                    : 0.023731

✅ SHAP EXPLAINABILITY ANALYSIS COMPLETE

⚠️  IMPORTANT NOTES:
   • Stage 1 SHAP excludes district_enc (categorical identifier)
   • Stage 1 model should be retrained WITHOUT district_enc
   • All visualizations show CLIMATE/ENVIRONMENTAL features only



In [49]:
# CELL 9: Generate Early Warning Explanation Text

print('\n' + '='*60)
print('CELL 9: Early Warning Report')
print('='*60)

# Find the target row again for Stage 1
target_row_stage1 = df[(df['district'] == 'Anuradhapura') & (df['month_num'] == 1) & (df['year'] == 2024)]

if len(target_row_stage1) == 0:
    target_row_stage1 = df[(df['district'] == 'Ampara') & (df['month_num'] == 1) & (df['year'] == 2024)]
    target_district = 'Ampara'
else:
    target_district = 'Anuradhapura'

if len(target_row_stage1) > 0:
    row_idx = target_row_stage1.index[0]
    
    # STAGE 1 INFO
    prob_stage1 = proba_stage1[row_idx]
    prob_pct = int(round(prob_stage1 * 100))
    
    # Determine risk level
    if prob_pct >= 66:
        risk_level = 'HIGH'
    elif prob_pct >= 33:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'
    
    # Get top 3 SHAP contributors (positive and negative)
    shap_row = shap_values_stage1[row_idx]
    top_indices = np.argsort(np.abs(shap_row))[::-1][:6]  # Get top 6 to ensure we have 3+ of each sign
    
    positive_contributors = []
    negative_contributors = []
    
    for idx in top_indices:
        if shap_row[idx] > 0:
            positive_contributors.append((feature_cols_stage1[idx], shap_row[idx]))
        elif shap_row[idx] < 0:
            negative_contributors.append((feature_cols_stage1[idx], shap_row[idx]))
    
    positive_contributors = positive_contributors[:3]
    negative_contributors = negative_contributors[:3]
    
    # STAGE 2 INFO
    target_row_stage2 = outbreak_rows[(outbreak_rows['district'] == target_district) & 
                                      (outbreak_rows['month_num'] == 1) & 
                                      (outbreak_rows['year'] == 2024)]
    
    if len(target_row_stage2) > 0:
        outbreak_idx = target_row_stage2.index[0]
        # Map to position in filtered outbreak_rows
        actual_idx = list(outbreak_rows.index).index(outbreak_idx)
        severity_pred_idx = int(stage2_preds[actual_idx])
        severity_mapping = {0: 'LOW', 1: 'MEDIUM', 2: 'HIGH'}
        severity_label = severity_mapping.get(severity_pred_idx, 'UNKNOWN')
    else:
        severity_label = 'N/A (No outbreak record)'
    
    # Generate report
    report = f"""
EARLY WARNING REPORT
=====================
District : {target_district}
Month    : January 2024

STAGE 1 — Outbreak Risk
Probability  : {prob_pct}%
Risk Level   : {risk_level}

Top contributing factors:
"""
    
    for feat, shap_val in positive_contributors:
        report += f"+ {feat} : +{shap_val:.2f} (increases risk)\n"
    
    for feat, shap_val in negative_contributors:
        report += f"- {feat} : {shap_val:.2f} (decreases risk)\n"
    
    report += f"""
STAGE 2 — Severity Estimate
Predicted Severity : {severity_label}

RECOMMENDATION: """
    
    if risk_level == 'HIGH' or (risk_level == 'MEDIUM' and severity_label in ['HIGH', 'MEDIUM']):
        report += "Deploy targeted surveillance and vaccination programs in high-risk areas."
    elif risk_level == 'MEDIUM':
        report += "Increase monitoring frequency and prepare rapid response teams."
    else:
        report += "Maintain routine monitoring and standard disease control measures."
    
    print(report)
    
    # Save report to file
    report_path = PROC_DIR / 'early_warning_report_anuradhapura_jan2024.txt'
    with open(report_path, 'w') as f:
        f.write(report)
    print(f'\n✅ Report saved to: {report_path}')
else:
    print('❌ Unable to generate report - target row not found')


CELL 9: Early Warning Report

EARLY WARNING REPORT
District : Anuradhapura
Month    : January 2024

STAGE 1 — Outbreak Risk
Probability  : 90%
Risk Level   : HIGH

Top contributing factors:
+ cos_month : +0.83 (increases risk)
+ rain_lag2 : +0.51 (increases risk)
+ humidity : +0.46 (increases risk)
- r3h : -0.82 (decreases risk)

STAGE 2 — Severity Estimate
Predicted Severity : MEDIUM

RECOMMENDATION: Deploy targeted surveillance and vaccination programs in high-risk areas.

✅ Report saved to: ..\data\processed\early_warning_report_anuradhapura_jan2024.txt


In [48]:
# PLOT 5: SHAP Summary Plot (Beeswarm) - Stage 2

print('\n' + '='*60)
print('PLOT 5: SHAP Summary Plot (Beeswarm) - Stage 2')
print('='*60)

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_for_plot,
    X_stage2,
    feature_names=stage2_feature_cols,
    show=False
)
plt.title('SHAP Summary — Stage 2 Severity Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = PLOT_DIR / 'stage2_shap_summary.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'✅ Saved: {plot_path}')


PLOT 5: SHAP Summary Plot (Beeswarm) - Stage 2
✅ Saved: ..\plots\08_shap\stage2_shap_summary.png


In [47]:
# PLOT 4: SHAP Feature Importance Bar Chart - Stage 2

print('\n' + '='*60)
print('PLOT 4: SHAP Feature Importance Bar Chart - Stage 2')
print('='*60)

# For multiclass, we need to plot class 1 (MEDIUM) or mean across classes
if isinstance(shap_values_stage2, list):
    # Use mean SHAP across all classes
    mean_shap = np.mean([np.abs(s) for s in shap_values_stage2], axis=0)
    shap_for_plot = mean_shap
else:
    shap_for_plot = shap_values_stage2

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_for_plot,
    X_stage2,
    feature_names=stage2_feature_cols,
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance — Stage 2 Severity Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = PLOT_DIR / 'stage2_shap_importance.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'✅ Saved: {plot_path}')


PLOT 4: SHAP Feature Importance Bar Chart - Stage 2
✅ Saved: ..\plots\08_shap\stage2_shap_importance.png


In [45]:
# PLOT 3: SHAP Waterfall for Single Prediction - Stage 1
# Note: Showing climate features only

print('\n' + '='*60)
print('PLOT 3: SHAP Waterfall Plot - Stage 1 (Anuradhapura Jan 2024)')
print('='*60)
print('(Showing climate/environmental features)')

# Find target row
target_row = df[(df['district'] == 'Anuradhapura') & (df['month_num'] == 1) & (df['year'] == 2024)]

if len(target_row) == 0:
    target_row = df[(df['district'] == 'Ampara') & (df['month_num'] == 1) & (df['year'] == 2024)]
    target_district = 'Ampara'
else:
    target_district = 'Anuradhapura'

if len(target_row) > 0:
    row_idx = target_row.index[0]
    
    # Create SHAP Explainer object for waterfall
    explainer = shap.LinearExplainer(stage1_lr_model, X_all_scaled)
    
    # Get prediction explanation
    sample_row = X_all_scaled.iloc[row_idx:row_idx+1]
    shap_sample = explainer.shap_values(sample_row)
    
    # Create waterfall with climate features only
    # Filter shap values and feature names
    climate_mask = [i for i, f in enumerate(feature_cols_stage1) if f != 'district_enc']
    shap_sample_climate = shap_sample[0, climate_mask]
    sample_row_climate = sample_row.iloc[:, climate_mask]
    
    plt.figure(figsize=(12, 8))
    shap.plots._waterfall.waterfall_legacy(
        explainer.expected_value,
        shap_sample_climate,
        sample_row_climate.values[0],
        feature_names=climate_feature_names,
        show=False
    )
    plt.title(f'SHAP Waterfall — {target_district} January 2024 (Climate Features)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    plot_path = PLOT_DIR / f'stage1_shap_waterfall_{target_district.lower()}_jan2024.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f'✅ Saved: {plot_path}')
    print(f'   District: {target_district}')
    print(f'   Prediction: {proba_stage1[row_idx]:.1%} outbreak probability')
else:
    print('❌ Target row not found')


PLOT 3: SHAP Waterfall Plot - Stage 1 (Anuradhapura Jan 2024)
(Showing climate/environmental features)
✅ Saved: ..\plots\08_shap\stage1_shap_waterfall_anuradhapura_jan2024.png
   District: Anuradhapura
   Prediction: 89.8% outbreak probability


In [46]:
# STAGE 2: SHAP Explanations (Random Forest)

print('\n' + '='*60)
print('STAGE 2: Computing SHAP Values (Random Forest)')
print('='*60)

# Filter to outbreak rows only
outbreak_mask = df['Outbreak status'] == 1
outbreak_rows = df[outbreak_mask].reset_index(drop=True)

print(f'Outbreak rows: {len(outbreak_rows)}')

# Prepare features for Stage 2
X_stage2 = outbreak_rows[stage2_feature_cols].fillna(0).astype(float)
print(f'✅ X_stage2 prepared: {X_stage2.shape}')

# Create TreeExplainer
print('Computing SHAP values for Random Forest (this may take a moment)...')
explainer_stage2 = shap.TreeExplainer(stage2_rf_model)
shap_values_stage2 = explainer_stage2.shap_values(X_stage2)

print(f'✅ SHAP values computed')
print(f'   Type: {type(shap_values_stage2)}')

# For multiclass, compute mean absolute SHAP 
if isinstance(shap_values_stage2, list):
    print(f'   Multiclass (list): {len(shap_values_stage2)} classes with shapes {[s.shape for s in shap_values_stage2]}')
    # Average across all classes and then take mean per feature
    mean_abs_per_class = [np.mean(np.abs(s), axis=0) for s in shap_values_stage2]
    mean_abs_shap_stage2 = np.mean(mean_abs_per_class, axis=0)
elif isinstance(shap_values_stage2, np.ndarray) and shap_values_stage2.ndim == 3:
    print(f'   Multiclass (3D array): shape {shap_values_stage2.shape}')
    # Shape: (n_samples, n_features, n_classes) -> take mean across samples and classes
    mean_abs_shap_stage2 = np.mean(np.abs(shap_values_stage2), axis=(0, 2))
else:
    print(f'   Standard case: shape {shap_values_stage2.shape}')
    mean_abs_shap_stage2 = np.mean(np.abs(shap_values_stage2), axis=0)

print(f'✅ Mean absolute SHAP per feature: {mean_abs_shap_stage2.shape}')
print(f'   Expected features: {len(stage2_feature_cols)}')

# Get predictions
stage2_preds = stage2_rf_model.predict(X_stage2)
print(f'✅ Predictions: {stage2_preds.shape}')
print(f'   Distribution: {np.bincount(stage2_preds)}')


STAGE 2: Computing SHAP Values (Random Forest)
Outbreak rows: 306
✅ X_stage2 prepared: (306, 21)
Computing SHAP values for Random Forest (this may take a moment)...
✅ SHAP values computed
   Type: <class 'numpy.ndarray'>
   Multiclass (3D array): shape (306, 21, 3)
✅ Mean absolute SHAP per feature: (21,)
   Expected features: 21
✅ Predictions: (306,)
   Distribution: [139 120  47]


In [44]:
# PLOT 2: SHAP Feature Importance Bar Chart - Stage 1
# Note: Excluding district_enc (identifier) from visualization

print('\n' + '='*60)
print('PLOT 2: SHAP Feature Importance Bar Chart - Stage 1')
print('='*60)
print('(Excluding district_enc - identifier feature)')

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values_climate, 
    X_all_climate, 
    feature_names=climate_feature_names, 
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance — Stage 1 (Climate/Environmental Features)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = PLOT_DIR / 'stage1_shap_importance.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'✅ Saved: {plot_path}')

# Compute mean absolute SHAP for CSV export (using climate features only)
mean_abs_shap_stage1 = np.mean(np.abs(shap_values_climate), axis=0)
print(f'✅ Mean absolute SHAP computed: {mean_abs_shap_stage1.shape} (climate features only)')
print(f'   Top 5 climate drivers:')
top_indices = np.argsort(mean_abs_shap_stage1)[::-1][:5]
for idx in top_indices:
    print(f'      {climate_feature_names[idx]:30s}: {mean_abs_shap_stage1[idx]:.6f}')


PLOT 2: SHAP Feature Importance Bar Chart - Stage 1
(Excluding district_enc - identifier feature)
✅ Saved: ..\plots\08_shap\stage1_shap_importance.png
✅ Mean absolute SHAP computed: (21,) (climate features only)
   Top 5 climate drivers:
      cos_month                     : 0.596302
      r3h                           : 0.556246
      lat                           : 0.316275
      humidity                      : 0.286263
      sin_month                     : 0.282171


In [43]:
# PLOT 1: SHAP Summary Plot (Beeswarm) - Stage 1
# Note: Excluding district_enc (identifier) from visualization

print('\n' + '='*60)
print('PLOT 1: SHAP Summary Plot (Beeswarm) - Stage 1')
print('='*60)
print('(Excluding district_enc - identifier feature)')

# Filter out district_enc for visualization
climate_feature_indices = [i for i, f in enumerate(feature_cols_stage1) if f != 'district_enc']
shap_values_climate = shap_values_stage1[:, climate_feature_indices]
X_all_climate = X_all.iloc[:, climate_feature_indices]
climate_feature_names = [feature_cols_stage1[i] for i in climate_feature_indices]

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values_climate, 
    X_all_climate, 
    feature_names=climate_feature_names, 
    show=False
)
plt.title('SHAP Summary — Stage 1 Outbreak Prediction (Climate/Environmental Features)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = PLOT_DIR / 'stage1_shap_summary.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'✅ Saved: {plot_path}')
print(f'   Features plotted: {len(climate_feature_names)} (excluding district_enc)')


PLOT 1: SHAP Summary Plot (Beeswarm) - Stage 1
(Excluding district_enc - identifier feature)
✅ Saved: ..\plots\08_shap\stage1_shap_summary.png
   Features plotted: 21 (excluding district_enc)


In [42]:
# STAGE 1: SHAP Explanations (Logistic Regression)

print('='*60)
print('STAGE 1: Computing SHAP Values (Logistic Regression)')
print('='*60)

# Create masker using Independent background
masker = shap.maskers.Independent(X_all_scaled)

# Create LinearExplainer (non-deprecated API)
explainer = shap.LinearExplainer(stage1_lr_model, masker)

# Compute SHAP values for full dataset
print('\nComputing SHAP values for full dataset...')
shap_values_stage1 = explainer.shap_values(X_all_scaled)

# Get prediction probabilities
proba_stage1 = stage1_lr_model.predict_proba(X_all_scaled)[:, 1]

print(f'✅ SHAP values computed: {shap_values_stage1.shape}')
print(f'✅ Probability array: {proba_stage1.shape}')
print(f'   Min probability: {proba_stage1.min():.4f}')
print(f'   Max probability: {proba_stage1.max():.4f}')
print(f'   Mean probability: {proba_stage1.mean():.4f}')

# Store for later use
expected_value_stage1 = explainer.expected_value
print(f'✅ Expected value (base value): {expected_value_stage1:.6f}')

STAGE 1: Computing SHAP Values (Logistic Regression)

Computing SHAP values for full dataset...
✅ SHAP values computed: (2400, 21)
✅ Probability array: (2400,)
   Min probability: 0.0374
   Max probability: 0.9386
   Mean probability: 0.5486
✅ Expected value (base value): 0.057994


In [41]:
import subprocess
import sys

try:
    import shap
    print('SHAP available')
except ImportError:
    print('Installing SHAP...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print('SHAP installed successfully')

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Define paths
PROJECT_ROOT = Path('D:\\Projects\\Research_Component')
DATA_DIR = PROJECT_ROOT / 'data'
PROC_DIR = DATA_DIR / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'
PLOT_DIR = PROJECT_ROOT / 'plots' / '08_shap'

# Create plots directory if it doesn't exist
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Load models (clean Stage 1 artifacts)
print('Loading models...')
stage1_lr_model = joblib.load(MODEL_DIR / 'stage1_lr_model.pkl')
stage1_scaler = joblib.load(MODEL_DIR / 'stage1_scaler.pkl')
stage1_feature_cols = joblib.load(MODEL_DIR / 'stage1_feature_cols.pkl')
stage2_rf_model = joblib.load(MODEL_DIR / 'stage2_rf_model.pkl')
stage2_label_encoder = joblib.load(MODEL_DIR / 'stage2_label_encoder.pkl')
stage2_feature_cols = joblib.load(MODEL_DIR / 'stage2_feature_cols.pkl')

print('✅ stage1_lr_model loaded')
print('✅ stage1_scaler loaded')
print('✅ stage1_feature_cols loaded')
print('✅ stage2_rf_model loaded')
print('✅ stage2_label_encoder loaded')
print('✅ stage2_feature_cols loaded')

# Load feature dataset
FEATURE_FILE = PROC_DIR / 'FMD_model_ready_main refined_final_dataset.csv'
df = pd.read_csv(FEATURE_FILE)
print(f'✅ Feature dataset loaded: {df.shape}')

# Use provided clean Stage 1 feature list (do NOT encode district)
feature_cols_stage1 = list(stage1_feature_cols)
print(f'\n✅ Using Stage 1 feature list from models/stage1_feature_cols.pkl ({len(feature_cols_stage1)} features)')
print(f'   Features: {feature_cols_stage1[:5]}...')

# Prepare X_all for Stage 1 (unscaled) — ensure columns exist
missing = [c for c in feature_cols_stage1 if c not in df.columns]
if missing:
    raise ValueError(f"Missing features in dataframe required by stage1_feature_cols.pkl: {missing}")

X_all = df[feature_cols_stage1].fillna(0).astype(float)
print(f'✅ X_all prepared: {X_all.shape}')

# Scale for Stage 1 model
X_all_scaled = stage1_scaler.transform(X_all)
X_all_scaled = pd.DataFrame(X_all_scaled, columns=feature_cols_stage1)
print(f'✅ X_all_scaled prepared: {X_all_scaled.shape}')

# Stage 2 feature columns already loaded
print(f'\n✅ Stage 2 feature columns: {len(stage2_feature_cols)} features')
print(f'   Features: {stage2_feature_cols[:5]}... (and {len(stage2_feature_cols)-5} more)')

print('\n' + '='*60)
print('SETUP COMPLETE — using clean Stage 1 artifacts (district_enc excluded)')
print('='*60)

SHAP available
Loading models...
✅ stage1_lr_model loaded
✅ stage1_scaler loaded
✅ stage1_feature_cols loaded
✅ stage2_rf_model loaded
✅ stage2_label_encoder loaded
✅ stage2_feature_cols loaded
✅ Feature dataset loaded: (2400, 26)

✅ Using Stage 1 feature list from models/stage1_feature_cols.pkl (21 features)
   Features: ['sin_month', 'cos_month', 'monsoon_phase_First_Inter_Monsoon', 'monsoon_phase_SW_Monsoon', 'monsoon_phase_Second_Inter_Monsoon']...
✅ X_all prepared: (2400, 21)
✅ X_all_scaled prepared: (2400, 21)

✅ Stage 2 feature columns: 21 features
   Features: ['sin_month', 'cos_month', 'monsoon_phase_First_Inter_Monsoon', 'monsoon_phase_SW_Monsoon', 'monsoon_phase_Second_Inter_Monsoon']... (and 16 more)

SETUP COMPLETE — using clean Stage 1 artifacts (district_enc excluded)


# 08 - SHAP Explainability for Two-Stage FMD Early Warning System

This notebook provides SHAP (SHapley Additive exPlanations) interpretability for both stages of the FMD early warning system.

## Objectives
1. Explain Stage 1 (Logistic Regression) outbreak predictions
2. Explain Stage 2 (Random Forest) severity classifications
3. Generate local explanations for specific predictions
4. Create an early warning report for Anuradhapura January 2024

In [38]:
# DEBUG: Check model feature expectations

print('='*60)
print('MODEL ARCHITECTURE DEBUG')
print('='*60)

# Check Stage 1 model
print('\nStage 1 Logistic Regression:')
print(f'  Model coefficients shape: {stage1_lr_model.coef_.shape}')
print(f'  Number of features model expects: {stage1_lr_model.n_features_in_}')

# Check Stage 1 scaler
print('\nStage 1 Scaler:')
print(f'  Scaler feature count: {stage1_scaler.n_features_in_}')
print(f'  Scaler features: {stage1_scaler.feature_names_in_}')

# Check our prepared data
print('\nOur Data:')
print(f'  X_all_scaled shape: {X_all_scaled.shape}')
print(f'  X_all_scaled columns: {list(X_all_scaled.columns)}')

# Check match
print('\n✅ COMPATIBILITY CHECK:')
if stage1_lr_model.n_features_in_ == X_all_scaled.shape[1]:
    print('  ✅ Model expects same number of features as provided')
else:
    print(f'  ❌ MISMATCH: Model expects {stage1_lr_model.n_features_in_}, got {X_all_scaled.shape[1]}')

# Show which features the model was trained on
print(f'\nFeatures model was trained with ({stage1_lr_model.n_features_in_}):')
if hasattr(stage1_scaler, 'feature_names_in_'):
    for i, feat in enumerate(stage1_scaler.feature_names_in_):
        print(f'  {i+1:2d}. {feat}')

MODEL ARCHITECTURE DEBUG

Stage 1 Logistic Regression:
  Model coefficients shape: (1, 22)
  Number of features model expects: 22

Stage 1 Scaler:
  Scaler feature count: 22
  Scaler features: ['sin_month' 'cos_month' 'monsoon_phase_First_Inter_Monsoon'
 'monsoon_phase_SW_Monsoon' 'monsoon_phase_Second_Inter_Monsoon'
 'monsoon_phase_NE_Monsoon' 'rainfall_mm' 'r3h' 'rfq' 'rain_lag1'
 'rain_lag2' 'rfq_lag1' 'lat' 'lon' 'humidity' 'wind_speed' 'temp_lag1'
 'humidity_lag1' 'wind_lag1' 'buffalo_density' 'livestock_density'
 'district_enc']

Our Data:
  X_all_scaled shape: (2400, 22)
  X_all_scaled columns: ['sin_month', 'cos_month', 'monsoon_phase_First_Inter_Monsoon', 'monsoon_phase_SW_Monsoon', 'monsoon_phase_Second_Inter_Monsoon', 'monsoon_phase_NE_Monsoon', 'rainfall_mm', 'r3h', 'rfq', 'rain_lag1', 'rain_lag2', 'rfq_lag1', 'lat', 'lon', 'humidity', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1', 'buffalo_density', 'livestock_density', 'district_enc']

✅ COMPATIBILITY CHECK:
  ✅

In [39]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.metrics import average_precision_score, roc_auc_score
import joblib

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(r'..\data\processed\FMD_model_ready_main refined_final_dataset.csv')

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# ── Define clean feature list — NO district_enc ───────────────────────────────
feature_cols = [
    'sin_month', 'cos_month',
    'monsoon_phase_First_Inter_Monsoon',
    'monsoon_phase_SW_Monsoon',
    'monsoon_phase_Second_Inter_Monsoon',
    'monsoon_phase_NE_Monsoon',
    'rainfall_mm', 'r3h', 'rfq',
    'rain_lag1', 'rain_lag2', 'rfq_lag1',
    'lat', 'lon',
    'humidity', 'wind_speed',
    'temp_lag1', 'humidity_lag1', 'wind_lag1',
    'buffalo_density', 'livestock_density'
]

print(f"\nFeature count : {len(feature_cols)}")  # should be 21
print("district_enc  : NOT included ✅")

# ── Verify all features exist in dataset ──────────────────────────────────────
missing = [f for f in feature_cols if f not in df.columns]
if missing:
    print(f"MISSING features: {missing}")
else:
    print("All features found in dataset ✅")

# ── Define X and y ────────────────────────────────────────────────────────────
X = df[feature_cols]
y = df['Outbreak status']

print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Outbreak rate: {y.mean()*100:.1f}%")

Dataset shape: (2400, 26)
Columns: ['year', 'month_num', 'district', 'PCODE', 'sin_month', 'cos_month', 'monsoon_phase_First_Inter_Monsoon', 'monsoon_phase_SW_Monsoon', 'monsoon_phase_Second_Inter_Monsoon', 'monsoon_phase_NE_Monsoon', 'rainfall_mm', 'r3h', 'rfq', 'rain_lag1', 'rain_lag2', 'rfq_lag1', 'Outbreak status', 'lat', 'lon', 'humidity', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1', 'buffalo_density', 'livestock_density']

Feature count : 21
district_enc  : NOT included ✅
All features found in dataset ✅

X shape : (2400, 21)
y shape : (2400,)
Outbreak rate: 12.8%


In [40]:
# ── Forward chaining splits ───────────────────────────────────────────────────
splits = [
    {'train_years': list(range(2017, 2022)), 'test_year': 2022},
    {'train_years': list(range(2017, 2023)), 'test_year': 2023},
    {'train_years': list(range(2017, 2024)), 'test_year': 2024},
]

results = []

for sp in splits:
    train_mask = df['year'].isin(sp['train_years'])
    test_mask  = df['year'] == sp['test_year']

    X_tr = X[train_mask]
    y_tr = y[train_mask]
    X_te = X[test_mask]
    y_te = y[test_mask]

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc  = sc.transform(X_te)

    lr = LogisticRegression(
        class_weight={0: 1, 1: 15},
        max_iter=1000,
        random_state=42
    )
    lr.fit(X_tr_sc, y_tr)

    y_pred  = lr.predict(X_te_sc)
    y_proba = lr.predict_proba(X_te_sc)[:, 1]

    results.append({
        'Test year' : sp['test_year'],
        'Precision' : round(precision_score(y_te, y_pred, zero_division=0), 3),
        'Recall'    : round(recall_score(y_te, y_pred), 3),
        'F1'        : round(f1_score(y_te, y_pred), 3),
        'PR-AUC'    : round(average_precision_score(y_te, y_proba), 3),
        'ROC-AUC'   : round(roc_auc_score(y_te, y_proba), 3),
        'Caught'    : f"{y_pred[y_te==1].sum()}/{y_te.sum()}"
    })

results_df = pd.DataFrame(results)
print("="*65)
print("CLEAN STAGE 1 MODEL — Forward Chaining Results")
print("="*65)
print(results_df.to_string(index=False))

# ── Average metrics ───────────────────────────────────────────────────────────
print("\nAVERAGE ACROSS ALL SPLITS:")
for col in ['Precision','Recall','F1','PR-AUC','ROC-AUC']:
    print(f"  {col:10s}: {results_df[col].mean():.3f}")

# ── Train FINAL model on 2017-2023, save ─────────────────────────────────────
print("\nTraining final model on 2017-2023...")

train_final_mask = df['year'] < 2024
X_final = X[train_final_mask]
y_final = y[train_final_mask]

sc_final = StandardScaler()
X_final_sc = sc_final.fit_transform(X_final)

lr_final = LogisticRegression(
    class_weight={0: 1, 1: 15},
    max_iter=1000,
    random_state=42
)
lr_final.fit(X_final_sc, y_final)

print(f"Final model features : {lr_final.n_features_in_}")
print(f"district_enc present : {'district_enc' in feature_cols}")

# ── Save clean models ─────────────────────────────────────────────────────────
joblib.dump(lr_final,    r'..\models\stage1_lr_model.pkl')
joblib.dump(sc_final,    r'..\models\stage1_scaler.pkl')
joblib.dump(feature_cols,r'..\models\stage1_feature_cols.pkl')

print("\n✅ Saved:")
print("   stage1_lr_model.pkl")
print("   stage1_scaler.pkl")
print("   stage1_feature_cols.pkl")

# ── Verify saved model ────────────────────────────────────────────────────────
loaded = joblib.load(r'..\models\stage1_lr_model.pkl')
print(f"\nVerification — loaded model expects: {loaded.n_features_in_} features ✅")

CLEAN STAGE 1 MODEL — Forward Chaining Results
 Test year  Precision  Recall    F1  PR-AUC  ROC-AUC Caught
      2022      0.152   0.889 0.260   0.270    0.692  32/36
      2023      0.061   0.917 0.115   0.192    0.757  11/12
      2024      0.394   0.949 0.556   0.659    0.845  74/78

AVERAGE ACROSS ALL SPLITS:
  Precision : 0.202
  Recall    : 0.918
  F1        : 0.310
  PR-AUC    : 0.374
  ROC-AUC   : 0.765

Training final model on 2017-2023...
Final model features : 21
district_enc present : False

✅ Saved:
   stage1_lr_model.pkl
   stage1_scaler.pkl
   stage1_feature_cols.pkl

Verification — loaded model expects: 21 features ✅
